# 🛣️ Chapter 04: Street Network Topology & pgRouting Graph Creation
## Constructing Navigable Graphs and Analyzing Orientation Entropies

### 🎯 The Mission: Why We Are Doing This
Spatial lines on a map are not a routing graph until intersections are topologically connected into **nodes** and **edges**. Using `pgrouting`, we convert Accra street centerlines into an intelligent network with travel impedance costs and heading azimuths.

---

### 🔑 Spatial DBA Keywords Decoded

| Keyword / Function | Plain-English Explanation | Production Note |
| :--- | :--- | :--- |
| **`pgr_createTopology`** | Splits lines at intersections, builds integer `source` and `target` IDs, and creates the `_vertices_pgr` table. | Mandatory first step for all routing algorithms. |
| **`pgr_analyzeGraph`** | Automated graph validator detecting dead-ends, gaps, and disconnected islands. | Network quality assurance tool. |
| **`ST_Azimuth(p1, p2)`** | Returns line heading angle in radians from North ($0	ext{ rad} = 	ext{North}$). | Used for street orientation rose charts. |

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

sql_network = """
DROP TABLE IF EXISTS accra_streets CASCADE;
CREATE TABLE accra_streets (
    id SERIAL PRIMARY KEY,
    name VARCHAR(100),
    highway VARCHAR(50),
    speed_kph INT DEFAULT 40,
    oneway VARCHAR(10) DEFAULT 'B',
    geom GEOMETRY(LineString, 32630)
);

INSERT INTO accra_streets (name, highway, speed_kph, oneway, geom)
VALUES 
    ('Liberation Road (N4)', 'primary', 70, 'B', ST_GeomFromText('LINESTRING(810000 615000, 812000 618000)', 32630)),
    ('Ring Road Central', 'primary', 60, 'B', ST_GeomFromText('LINESTRING(808000 614000, 810000 615000)', 32630)),
    ('Oxford Street (Osu)', 'tertiary', 30, 'B', ST_GeomFromText('LINESTRING(810000 615000, 810500 613500)', 32630)),
    ('Cantonments Road', 'residential', 30, 'B', ST_GeomFromText('LINESTRING(810500 613500, 811500 614000)', 32630)),
    ('Airport Bypass', 'secondary', 50, 'FT', ST_GeomFromText('LINESTRING(812000 618000, 811500 614000)', 32630));

ALTER TABLE accra_streets ADD COLUMN IF NOT EXISTS source INT;
ALTER TABLE accra_streets ADD COLUMN IF NOT EXISTS target INT;
ALTER TABLE accra_streets ADD COLUMN IF NOT EXISTS cost_len DOUBLE PRECISION;
ALTER TABLE accra_streets ADD COLUMN IF NOT EXISTS cost_sec DOUBLE PRECISION;
ALTER TABLE accra_streets ADD COLUMN IF NOT EXISTS reverse_cost_sec DOUBLE PRECISION;

UPDATE accra_streets
SET cost_len = ST_Length(geom),
    cost_sec = ST_Length(geom) / (speed_kph * 1000.0 / 3600.0),
    reverse_cost_sec = CASE 
        WHEN oneway = 'FT' THEN -1.0
        ELSE ST_Length(geom) / (speed_kph * 1000.0 / 3600.0)
    END;

SELECT pgr_createTopology('accra_streets', 1.0, 'geom', 'id', 'source', 'target');
"""

execute_sql(sql_network)
print("✅ Road network graph built and topology initialized.")

In [ ]:
# Inspect Azimuth Headings & Network Edge Attributes
df_azimuth = query_df("""
SELECT id, name, highway, ROUND((DEGREES(ST_Azimuth(ST_StartPoint(geom), ST_EndPoint(geom)))::NUMERIC % 180), 1) AS orientation_deg, ROUND(cost_len::NUMERIC, 1) AS length_m, ROUND(cost_sec::NUMERIC, 1) AS travel_time_sec
FROM accra_streets;
""")

df_azimuth

In [ ]:
# Interactive Map Preview of Street Network Edges
gdf_streets = query_gdf("""
SELECT id, name, highway, speed_kph, geom FROM accra_streets;
""")

preview_map(gdf_streets, tooltip_cols=["name", "highway", "speed_kph"])